# Product-Inventory Problem Test

In [1]:
import lropt

In [2]:
import cvxpy as cp

import matplotlib.pyplot as plt
import numpy as np
import numpy.random as npr
import numpy.testing as npt
import scipy as sc
import torch
from sklearn.model_selection import train_test_split

In [39]:
seed = 15
np.random.seed(seed)
kappa = -0.01
n = 2 # number of time periods

# family parameter (production costs)
dist = [1-0.5*np.sin(np.pi*(t-1)/12) for t in range(1, n+1)]
sig = np.random.rand(n, n)
sig = np.dot(sig, sig.T) # to ensure symmetric positive definiteness
y_data = np.random.multivariate_normal(dist, sig, 10)

In [40]:
# uncertainty distribution
def gen_demand_intro(N, seed):
    np.random.seed(seed)
    theta = 0.2
    d_hat = [1000*(1+0.5*np.sin(np.pi*(t-1)/12)) for t in range(1, n+1)] # nominal demand
    d_train = d_hat * np.random.uniform(1-theta, 1+theta, (N, n))
    print(d_hat)
    return d_train

In [41]:
# initialize problem
P = np.full(n, 567) # max production capacity at each time period
Q = 1100 # max production capacity for all periods
V_min = 500 # minimum inventory
V_max = 2000 # maximum inventory
v_1 = 2000 # initial inventory
data = gen_demand_intro(600, seed=15) # demand

[1000.0, 1129.4095225512604]


## Problem with Certain Demand

In [50]:
print(dist) # costs
print(data[0]) # demand

[1.0, 0.8705904774487396]
[1139.52707891  984.3463225 ]


In [46]:
x = cp.Variable(n)
u = data[0]
objective = cp.Minimize(x @ dist)
constraints = [cp.sum(x) <= Q, x >=0, x<= P]
for t in range(n):
    b = np.concatenate((np.ones(t+1), np.zeros(n-t-1)))
    constraints.append(x @ b - u @ b >= V_min - v_1)
    constraints.append(x @ b - u @ b <= V_max - v_1)

prob = cp.Problem(objective, constraints)

In [47]:
prob.solve()

550.4982029879427

In [51]:
x.value

array([ 56.87340635, 566.99999531])

Above results have been checked (matches intuitive optimal solutions).

## Problem with Uncertain Demand

In [ ]:
# now use all of data, instead of just one for u



## Using LROPT

In [16]:
y = lropt.Parameter(n, data=y_data)
u = lropt.UncertainParameter(n, uncertainty_set=lropt.Box(data=data))
x = cp.Variable(n)
objective = cp.Minimize(x @ y)
constraints = [cp.sum(x) <= Q, x >= 0, x <= P]
for t in range(n):
    b = np.concatenate((np.ones(t+1), np.zeros(n-t-1)))
    constraints.append(x @ b - u @ b >= V_min - v_1)
    constraints.append(x @ b - u @ b <= V_max - v_1)
prob = lropt.RobustProblem(objective, constraints)

In [17]:
test_p = 0.1
s = 5
train, _ = train_test_split(data, test_size=int(
    data.shape[0]*test_p), random_state=s)

init = sc.linalg.sqrtm(sc.linalg.inv(np.cov(train.T)))
init_bval= -init@np.mean(train, axis=0)
np.random.seed(15)
initn = np.random.rand(n, n) + 0.1*init + 0.5*np.eye(n)
init_bvaln = -initn@(np.mean(train, axis=0) - 0.3*np.ones(n))

In [18]:
# Train A and b
result = prob.train(lr=0.001, num_iter=100, momentum=0.8, # increase numbe of iterations
                    optimizer="SGD",
                    seed=s, init_A=initn, init_b=init_bvaln,
                    init_lam=0.5, init_mu=0.01,
                    mu_multiplier=1.001, init_alpha=0., test_percentage=test_p, kappa=kappa,
                    parallel = True, random_init=True, num_random_init=2, position = True)


run 0: test value N/A, violations N/A:   0%|          | 0/100 [00:00<?, ?it/s]/Users/annieliang/anaconda3/envs/lropt/lib/python3.11/site-packages/threadpoolctl.py:1010: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
/Users/annieliang/anaconda3/envs/lropt/lib/python3.11/site-packages/threadpoolctl.py:1010: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashe

In [19]:
results_df = result.df
df_test = result.df_test

In [21]:
result.weights

[tensor([[ 1.5410, -0.2934],
         [-2.1788,  0.5684]], dtype=torch.float64),
 tensor([-1.0845, -1.3986], dtype=torch.float64)]

In [34]:
result.var_values # x

(tensor([[8.2606e-10, 6.4348e-10],
         [7.7332e-10, 6.6753e-10],
         [2.4499e-10, 1.5366e-09],
         [2.6300e-11, 3.7549e-11],
         [1.3896e-09, 4.5678e-10],
         [2.9942e-10, 2.8647e-10],
         [1.0375e-10, 7.2896e-11],
         [2.2007e-11, 3.2385e-11],
         [1.0632e-09, 8.2701e-10],
         [1.0177e-09, 6.0385e-10]], dtype=torch.float64, requires_grad=True),)

In [30]:
result.obj

9.90310381658945e-10